In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_auc_score, ConfusionMatrixDisplay)
from imblearn.over_sampling import SMOTE
from sklearn.tree import plot_tree

: 

In [ ]:
df = pd.read_csv('full_gen_data.csv')
print(df.shape)
df.head()

In [ ]:
print("Missing values per column:\n", df.isna().sum())
print("\nDuplicate rows:", df.duplicated().sum())
df = df.drop_duplicates()
df = df.dropna()

In [ ]:
sns.countplot(x='label', data=df)
plt.title('Purchase vs No Purchase after Campaign')
plt.xlabel('0 = No Purchase, 1 = Purchase')
plt.show()
print(df['label'].value_counts(normalize=True))

In [ ]:
df['discount'] = 1 - df['ratio']
sns.boxplot(x='label', y='discount', data=df)
plt.title('Discount Depth vs Purchase Outcome')
plt.show()

In [ ]:
for promo in ['promo1', 'promo2']:
    print(df.groupby(promo)['label'].mean())

In [ ]:
cat_conv = df.groupby('category')['label'].mean().sort_values(ascending=False)
sns.barplot(x=cat_conv.values, y=cat_conv.index)
plt.title('Conversion Rate by Category')
plt.xlabel('Conversion Rate')
plt.show()

In [ ]:
# can contain overlapping info so we do heatmap
numeric_cols = ['sales','regular_price','current_price','ratio','discount',
                 'cost','promo1','promo2','label']
sns.heatmap(df[numeric_cols].corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap')
plt.show()

In [ ]:
df['retailweek'] = pd.to_datetime(df['retailweek'])
df['month'] = df['retailweek'].dt.month
df['quarter'] = df['retailweek'].dt.quarter
df_model = df.drop(columns=['article', 'article.1', 'customer_id', 'retailweek', 'ratio'])

In [ ]:
cat_cols = ['country', 'productgroup', 'category', 'style', 'sizes', 'gender']
df_model = pd.get_dummies(df_model, columns=cat_cols, drop_first=True)

In [ ]:
X = df_model.drop(columns=['label'])
y = df_model['label']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(X_train, y_train)
print("Before:", y_train.value_counts().to_dict())
print("After:", y_train_res.value_counts().to_dict())

In [ ]:
scaler = StandardScaler() # (original value-mean) / standard deviation
X_train_scaled = scaler.fit_transform(X_train_res)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# sigmoid function
# accuracy = correct predictions / total predictions
# recall = true positive / true positive + false negative
# precision = true positive / true positive + false positive
# ROC-AUC = measures how well the model separates positive and negative cases across different classification thresholds.
log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train_scaled, y_train_res)
y_pred_lr = log_reg.predict(X_test_scaled)
print(classification_report(y_test, y_pred_lr))
print("ROC-AUC:", roc_auc_score(y_test, log_reg.predict_proba(X_test_scaled)[:,1]))

In [ ]:
rf = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42)
rf.fit(X_train_res, y_train_res)
y_pred_rf = rf.predict(X_test)
print(classification_report(y_test, y_pred_rf))
print("ROC-AUC:", roc_auc_score(y_test, rf.predict_proba(X_test)[:,1]))

In [ ]:
lr_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=1000, random_state=42))
])
rf_for_ensemble = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42)

ensemble = VotingClassifier(
    estimators=[('lr', lr_pipe), ('rf', rf_for_ensemble)],
    voting='soft' # avg of both probabilities
)
ensemble.fit(X_train_res, y_train_res)
y_pred_ens = ensemble.predict(X_test)
print(classification_report(y_test, y_pred_ens))
print("ROC-AUC:", roc_auc_score(y_test, ensemble.predict_proba(X_test)[:,1]))

In [ ]:
for name, y_pred in [('Logistic Regression', y_pred_lr),
                      ('Random Forest', y_pred_rf),
                      ('Ensemble', y_pred_ens)]:
    cm = confusion_matrix(y_test, y_pred)
    ConfusionMatrixDisplay(cm).plot()
    plt.title(name)
    plt.show()

In [ ]:
# strongest features used by the model to make predictions
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
importances.head(10).plot(kind='barh')
plt.title('Top 10 Feature Importances (Random Forest)')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
coeffs = pd.Series(log_reg.coef_[0], index=X.columns).sort_values()
coeffs.tail(10).plot(kind='barh', color='green')
plt.title('Top 10 Features Increasing Purchase Probability')
plt.show()
coeffs.head(10).plot(kind='barh', color='red')
plt.title('Top 10 Features Decreasing Purchase Probability')
plt.show()

In [ ]:
plt.figure(figsize=(20, 10))
plot_tree(rf.estimators_[0], feature_names=X.columns, class_names=['No Purchase','Purchase'],
          filled=True, max_depth=3, fontsize=8)
plt.title('One Tree from the Random Forest (first 3 levels)')
plt.show()